# CKD Multi-Seed Validation

This notebook checks whether the near-perfect CKD result depends on a single
train/test split. The Random Forest baseline is re-run across 3 different
random seeds (42, 7, 123) with everything else held fixed — same model
hyperparameters, same 80/20 stratified split ratio, same preprocessing.

If the result is genuine (not a lucky split), accuracy should stay high and
consistent across all three seeds, with small natural variance.

Output: `results/week6/ckd_multiseed_validation.csv`

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

os.makedirs("../../results/week6", exist_ok=True)

In [2]:
df = pd.read_csv("../../data/processed/chronic_kidney_disease_processed.csv")

X = df.drop("class", axis=1)
y = df["class"]

print(df.shape)

(400, 25)


## Run Random Forest across 3 seeds

Each seed controls which 80 patients end up in the test set. The model
itself (`RandomForestClassifier(random_state=42)`) is kept fixed across all
three runs — only the train/test split changes.

In [3]:
seeds = [42, 7, 123]
rows = []

for seed in seeds:

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rows.append({
        "Seed": seed,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

results = pd.DataFrame(rows)
results

,Seed,Accuracy,Precision,Recall,F1
0,42,0.9750,1.000000,0.933333,0.965517
1,7,0.9875,0.967742,1.000000,0.983607
2,123,0.9750,1.000000,0.933333,0.965517


## Summary — mean and standard deviation across seeds

In [4]:
summary_cols = ["Accuracy", "Precision", "Recall", "F1"]

mean_row = {"Seed": "Mean", **results[summary_cols].mean().to_dict()}
std_row = {"Seed": "Std", **results[summary_cols].std().to_dict()}

final = pd.concat([results, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print(final.round(4))

   Seed  Accuracy  Precision  Recall      F1
0    42    0.9750     1.0000  0.9333  0.9655
1     7    0.9875     0.9677  1.0000  0.9836
2   123    0.9750     1.0000  0.9333  0.9655
3  Mean    0.9792     0.9892  0.9556  0.9715
4   Std    0.0072     0.0186  0.0385  0.0104


In [5]:
final.to_csv(
    "../../results/week6/ckd_multiseed_validation.csv",
    index=False
)

print("Saved to results/week6/ckd_multiseed_validation.csv")

Saved to results/week6/ckd_multiseed_validation.csv


### Observation

Random Forest was evaluated on the CKD dataset across three different random
seeds controlling the train/test split. Accuracy remained consistently high
(mean ≈ 0.98, std < 0.01) rather than dropping under different splits, which
supports that the near-perfect result on the original single split reflects
the genuine separability of this dataset rather than a leakage artifact or a
single favorable split.